# 02 — Tajribalar: nima sinab ko'rdik va nima o'rgandik
Har bir katak bitta yondashuv, hammasi **bir xil 5-fold stratified CV** (fold seed = 42) bilan baholanadi, natijalar oxirida
jadvalga yig'iladi va `../outputs/experiments.csv` ga yoziladi.
Featurelar `src/train.py` dagi funksiyalardan olinadi, shuning uchun bu notebook va yakuniy model aynan bir xil kodni ishlatadi.

`notebooks/` papkasidan ishga tushiring. Ish vaqti ≈ 15–20 daqiqa (GRU siz), GRU bilan GPU'da +15 daqiqa.

In [1]:
import sys, os, time, warnings, numpy as np, pandas as pd, lightgbm as lgb
sys.path.insert(0, '../src'); warnings.filterwarnings('ignore')
from train import features_windows, features_histograms, features_relative, LGB_PARAMS, BURST_CUTOFF_DAYS
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import rankdata

DATA, OUT, CACHE = '../data', '../outputs', '../outputs/cache'
os.makedirs(CACHE, exist_ok=True)
tr = pd.read_csv(f'{DATA}/train_signals.csv', parse_dates=['signal_sanasi'])
y = tr.eskalatsiya.values
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(tr, y))
RESULTS = []                                   # (kalit, nomi, AUC, xulosa)
import re
_K = {'1.': 'agg', '2.': 'agg_hist', '3.': 'all853', '4.': 'lr853', '6a.': 'lr60', '7.': 'catboost', '8.': 'mil', '9.': 'gru', '11.': 'floor'}
def _key(name):
    n = name.split(' ')[0]
    if n == '5.': return 'top' + re.search(r'Top-(\d+)', name).group(1)
    if n == '6b.': return 'blend' + re.search(r'LightGBM (\d)\.(\d)', name).group(2)
    if n == '10.': return 'hp_' + re.sub(r'[^a-z0-9]+', '', name.split(',')[-1].lower())
    return _K[n]
def record(name, auc, note):
    RESULTS.append((_key(name), name, round(auc, 4), note)); print(f'{name:55s} AUC = {auc:.4f}')
    pd.DataFrame(RESULTS, columns=['key', 'Yondashuv', 'CV ROC-AUC', 'Xulosa']).to_csv(f'{OUT}/experiments.csv', index=False)

def cv_lgb(X, early_stop=True, n_estimators=1500, params=None):
    """LightGBM 5-fold CV. early_stop=True — feature to'plamlarini solishtirish uchun; False — yakuniy retsept."""
    p = {**LGB_PARAMS, **(params or {})}; oof = np.zeros(len(y))
    for tri, vai in FOLDS:
        m = lgb.LGBMClassifier(n_estimators=5000 if early_stop else n_estimators, random_state=42, **p)
        if early_stop:
            m.fit(X.iloc[tri], y[tri], eval_set=[(X.iloc[vai], y[vai])], eval_metric='auc',
                  callbacks=[lgb.early_stopping(300, verbose=False)])
        else:
            m.fit(X.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(X.iloc[vai])[:, 1]
    return oof

def cv_lr(X, C=0.01):
    oof = np.zeros(len(y))
    for tri, vai in FOLDS:
        m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=C, max_iter=3000))
        oof[vai] = m.fit(X.iloc[tri], y[tri]).predict_proba(X.iloc[vai])[:, 1]
    return oof

## 0. Featurelarni bir marta hisoblab, keshlash

In [2]:
def cached(name, fn):
    p = f'{CACHE}/{name}.parquet'
    if os.path.exists(p): return pd.read_parquet(p)
    t = time.time(); F = fn(tr, f'{DATA}/train_transactions.parquet'); F.to_parquet(p)
    print(f'{name}: {F.shape[1]} feature, {time.time()-t:.0f}s'); return F

F_win = cached('windows', features_windows)        # vaqt oynalari bo'yicha agregatlar
F_hist = cached('histograms', features_histograms) # summa taqsimoti shakli
F_rel = cached('relative', features_relative)      # turlar orasidagi nisbiy farqlar
clean = lambda X: X.loc[:, ~X.columns.duplicated()].replace([np.inf, -np.inf], np.nan).astype('float32')
X_win, X_wh, X_all = clean(F_win), clean(pd.concat([F_win, F_hist], axis=1)), clean(pd.concat([F_win, F_hist, F_rel], axis=1))
print(X_win.shape, X_wh.shape, X_all.shape)

(14000, 387) (14000, 536) (14000, 853)


## 1. Bazaviy: oddiy agregatlar (count, mean, std, ulushlar) + LightGBM
Ko'pchilik jamoalar boshlaydigan nuqta.

In [3]:
oof = cv_lgb(X_win)
record('1. Agregatlar (oynalar) + LightGBM', roc_auc_score(y, oof), 'bazaviy nuqta')

1. Agregatlar (oynalar) + LightGBM                      AUC = 0.6271


## 2. + summa gistogrammalari va kvantillar
EDA 4-bo'lim: sinflar farqi taqsimotning *shaklida*, o'rtachasida emas.

In [4]:
oof = cv_lgb(X_wh)
record('2. + gistogramma/kvantil featurelari', roc_auc_score(y, oof), f'bazaviyga nisbatan {roc_auc_score(y, oof) - RESULTS[0][2]:+.3f}')

2. + gistogramma/kvantil featurelari                    AUC = 0.6297


## 3. + turlar orasidagi nisbiy farqlar
EDA 7-bo'lim: bank o'tkazmasi − naqd / karta farqi eng kuchli signal.

In [5]:
oof_all = cv_lgb(X_all)
record('3. + nisbiy tur/yo\'nalish farqlari (853 feature)', roc_auc_score(y, oof_all), f'oldingi qadamga nisbatan {roc_auc_score(y, oof_all) - RESULTS[1][2]:+.3f}')

3. + nisbiy tur/yo'nalish farqlari (853 feature)        AUC = 0.6520


## 4. Chiziqli model (logistik regressiya) — signal qanchalik chiziqli?
Agar boosting chiziqli modeldan ancha o'zsa, ma'lumotda murakkab o'zaro ta'sirlar bor. O'zmasa — signal oddiy va zaif.

In [6]:
oof = cv_lr(X_all)
record('4. Logistik regressiya (853 feature)', roc_auc_score(y, oof), f'LightGBM (3-qadam) bilan farq {roc_auc_score(y, oof) - RESULTS[2][2]:+.3f}')

4. Logistik regressiya (853 feature)                    AUC = 0.6198


## 5. Feature tanlash: 853 → 60
Featurelarning ko'pi shovqin. LightGBM gain bo'yicha saralab, eng yaxshi 60 tasini qoldiramiz (yakuniy modeldagi usul).

In [7]:
imp = pd.Series(0.0, index=X_all.columns)
for tri, vai in FOLDS:
    m = lgb.LGBMClassifier(n_estimators=5000, random_state=42, **LGB_PARAMS)
    m.fit(X_all.iloc[tri], y[tri], eval_set=[(X_all.iloc[vai], y[vai])], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    imp += pd.Series(m.booster_.feature_importance('gain'), index=X_all.columns)
imp = imp.sort_values(ascending=False)
for k in [30, 60, 120]:
    oof = cv_lgb(X_all[list(imp.index[:k])], early_stop=False)
    record(f'5. Top-{k} feature + LightGBM (1500 daraxt)', roc_auc_score(y, oof), 'shovqinli featurelarni olib tashlash')
TOP = list(imp.index[:60]); X_top = X_all[TOP]
oof_lgb_top = cv_lgb(X_top, early_stop=False)

5. Top-30 feature + LightGBM (1500 daraxt)              AUC = 0.6567
5. Top-60 feature + LightGBM (1500 daraxt)              AUC = 0.6618
5. Top-120 feature + LightGBM (1500 daraxt)             AUC = 0.6604


## 6. Logistik regressiya top-60 da va rank-blend

In [8]:
oof_lr_top = cv_lr(X_top)
record('6a. Logistik regressiya (top-60)', roc_auc_score(y, oof_lr_top), '')
for w in [0.5, 0.7, 0.9]:
    bl = w * rankdata(oof_lgb_top) + (1 - w) * rankdata(oof_lr_top)
    record(f'6b. Blend LightGBM {w:.1f} / LogReg {1-w:.1f}', roc_auc_score(y, bl), 'yakuniy retsept' if abs(w - 0.7) < 1e-9 else '')

6a. Logistik regressiya (top-60)                        AUC = 0.6476
6b. Blend LightGBM 0.5 / LogReg 0.5                     AUC = 0.6588
6b. Blend LightGBM 0.7 / LogReg 0.3                     AUC = 0.6610
6b. Blend LightGBM 0.9 / LogReg 0.1                     AUC = 0.6619


## 7. CatBoost (alternativ boosting)

In [9]:
try:
    from catboost import CatBoostClassifier
    oof = np.zeros(len(y))
    for tri, vai in FOLDS:
        m = CatBoostClassifier(iterations=3000, learning_rate=0.02, depth=4, l2_leaf_reg=10, rsm=0.3, eval_metric='AUC',
                               od_type='Iter', od_wait=300, verbose=0, random_seed=42)
        m.fit(X_all.iloc[tri], y[tri], eval_set=(X_all.iloc[vai], y[vai])); oof[vai] = m.predict_proba(X_all.iloc[vai])[:, 1]
    record('7. CatBoost (853 feature)', roc_auc_score(y, oof), f'LightGBM (3-qadam) bilan farq {roc_auc_score(y, oof) - RESULTS[2][2]:+.3f}')
except ImportError:
    print('catboost o\'rnatilmagan — pip install catboost'); record('7. CatBoost (853 feature)', float('nan'), 'ishga tushirilmadi')

7. CatBoost (853 feature)                               AUC = 0.6469


## 8. Tranzaksiya darajasidagi model (multiple-instance)
Har bir tranzaksiyaga uning alertining label'ini berib o'rgatamiz, keyin alert bo'yicha o'rtacha olamiz.
Agar alohida tranzaksiyalar ma'lumot bersa, bu ishlaydi.

In [10]:
tx = pd.read_parquet(f'{DATA}/train_transactions.parquet').merge(tr, on='signal_id')
tx['d'] = (tx.signal_sanasi - tx.tranzaksiya_vaqti).dt.total_seconds() / 86400
tx = tx[tx.d > BURST_CUTOFF_DAYS].sort_values(['signal_id', 'tranzaksiya_vaqti'])
g = tx.groupby('signal_id')
tx['gap'] = g.tranzaksiya_vaqti.diff().dt.total_seconds() / 3600
tx['prev_amt'] = g.miqdor_indeksi.shift(); tx['out'] = (tx.kirim_chiqim == 'chiqim').astype(int)
tx['typ'] = tx.tranzaksiya_turi.astype('category').cat.codes; tx['hour'] = tx.tranzaksiya_vaqti.dt.hour
cols = ['miqdor_indeksi', 'd', 'gap', 'prev_amt', 'out', 'typ', 'hour']
sid = tr.signal_id.values; oof = np.zeros(len(y))
for tri, vai in FOLDS:
    m = tx.signal_id.isin(set(sid[tri]))
    clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=1000,
                             subsample=0.3, subsample_freq=1, verbose=-1)
    clf.fit(tx.loc[m, cols], tx.loc[m, 'eskalatsiya'])
    p = pd.Series(clf.predict_proba(tx.loc[~m, cols], raw_score=True), index=tx.loc[~m, 'signal_id'].values).groupby(level=0).mean()
    oof[vai] = p.reindex(sid[vai]).fillna(p.mean()).values
record('8. Tranzaksiya darajasidagi model (MIL)', roc_auc_score(y, oof), 'har tranzaksiyaga alert label\'i, alert bo\'yicha o\'rtacha')
del tx

8. Tranzaksiya darajasidagi model (MIL)                 AUC = 0.5543


## 9. Ketma-ketlik modeli: bi-GRU + attention (PyTorch, ixtiyoriy)
Jamoaning dastlabki yondashuvi. Bu yerda tuzatilgan versiya: attention mask, tur uchun embedding, burst artefakti chiqarilgan,
kuchli regularizatsiya, early stopping. `RUN_GRU = True` va GPU bo'lsa ~15 daqiqa.

In [11]:
RUN_GRU = False
try:
    import torch, torch.nn as nn
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
if RUN_GRU and HAVE_TORCH:
    from torch.utils.data import DataLoader
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', dev)
    tx = pd.read_parquet(f'{DATA}/train_transactions.parquet').merge(tr, on='signal_id')
    tx['d'] = (tx.signal_sanasi - tx.tranzaksiya_vaqti).dt.total_seconds() / 86400
    tx = tx[tx.d > BURST_CUTOFF_DAYS].sort_values(['signal_id', 'tranzaksiya_vaqti'])
    tx['hour'] = tx.tranzaksiya_vaqti.dt.hour / 23; tx['dn'] = tx.d / 180
    tx['out'] = (tx.kirim_chiqim == 'chiqim').astype('float32'); tx['typ'] = tx.tranzaksiya_turi.astype('category').cat.codes
    tx['amt'] = (tx.miqdor_indeksi - tx.miqdor_indeksi.mean()) / tx.miqdor_indeksi.std()
    seqs = {s: (g[['amt', 'dn', 'hour', 'out']].values.astype('float32'), g.typ.values.astype('int64')) for s, g in tx.groupby('signal_id')}
    def collate(batch):
        L = torch.tensor([len(seqs[s][0]) for s, _ in batch]); T = int(L.max())
        X = torch.zeros(len(batch), T, 4); K = torch.zeros(len(batch), T, dtype=torch.long)
        for i, (s, _) in enumerate(batch):
            a, k = seqs[s]; X[i, :len(a)] = torch.tensor(a); K[i, :len(k)] = torch.tensor(k)
        return X, K, L, torch.tensor([l for _, l in batch], dtype=torch.float32)
    class GRUNet(nn.Module):
        def __init__(s, h=32):
            super().__init__(); s.emb = nn.Embedding(4, 4)
            s.gru = nn.GRU(8, h, batch_first=True, bidirectional=True); s.att = nn.Linear(2*h, 1)
            s.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(2*h, 16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 1))
        def forward(s, x, k, L):
            z = torch.cat([x, s.emb(k)], -1)
            packed = nn.utils.rnn.pack_padded_sequence(z, L.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = s.gru(packed); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
            mask = torch.arange(o.size(1), device=o.device)[None, :] < L.to(o.device)[:, None]      # padding uchun mask
            w = torch.softmax(s.att(o).squeeze(-1).masked_fill(~mask, -1e9), 1)
            return s.fc((o * w.unsqueeze(-1)).sum(1)).squeeze(-1)
    oof = np.zeros(len(y)); sid = tr.signal_id.values
    for f, (tri, vai) in enumerate(FOLDS):
        net = GRUNet().to(dev); opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-3)
        lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - y.mean()) / y.mean()]).to(dev))
        trl = DataLoader(list(zip(sid[tri], y[tri])), batch_size=128, shuffle=True, collate_fn=collate)
        val = DataLoader(list(zip(sid[vai], y[vai])), batch_size=256, collate_fn=collate)
        best, best_p, bad = 0, None, 0
        for ep in range(30):
            net.train()
            for X, K, L, t in trl:
                opt.zero_grad(); loss = lossf(net(X.to(dev), K.to(dev), L), t.to(dev)); loss.backward(); opt.step()
            net.eval(); p = []
            with torch.no_grad():
                for X, K, L, t in val: p.append(torch.sigmoid(net(X.to(dev), K.to(dev), L)).cpu().numpy())
            p = np.concatenate(p); a = roc_auc_score(y[vai], p)
            if a > best: best, best_p, bad = a, p, 0
            else: bad += 1
            if bad >= 5: break
        oof[vai] = best_p; print(f'fold {f}: best AUC {best:.4f} (epoch {ep+1-bad})')
    record('9. bi-GRU + attention (tuzatilgan)', roc_auc_score(y, oof), 'tartib ma\'lumoti zaif; overfitting')
else:
    # jamoaning dastlabki GRU tajribasi (80/20 split, 100 epoch) — natija uning asl logidan o'qiladi
    import re
    log = open(f'{OUT}/gru_training.log', encoding='utf-8').read()
    ep = [(int(e), float(l), float(a)) for e, l, a in re.findall(r'Epoch (\d+)/\d+ yakunlandi \| Train Loss: ([\d.]+) \| Valid AUC: ([\d.]+)', log)]
    best = max(ep, key=lambda t: t[2])
    record('9. bi-GRU + attention (jamoaning dastlabki tajribasi, 80/20 split)', best[2],
           f'eng yaxshi {best[0]}-epochda; oxirgi epoch valid AUC {ep[-1][2]:.3f}, train loss {ep[-1][1]:.3f}')

9. bi-GRU + attention (jamoaning dastlabki tajribasi, 80/20 split) AUC = 0.6264


## 10. Giperparametrlar (top-60 da)

In [12]:
for name, n, p in [('num_leaves=4', 1500, {'num_leaves': 4}), ('num_leaves=15', 700, {'num_leaves': 15}),
                   ('extra_trees', 1200, {'extra_trees': True}), ('min_child_samples=300', 1500, {'min_child_samples': 300}),
                   ('colsample=0.5', 1500, {'colsample_bytree': 0.5}), ('max_depth=3', 1200, {'num_leaves': 8, 'max_depth': 3})]:
    oof = cv_lgb(X_top, early_stop=False, n_estimators=n, params=p)
    record(f'10. LightGBM top-60, {name}', roc_auc_score(y, oof), f'asosiy parametrlarga nisbatan {roc_auc_score(y, oof) - roc_auc_score(y, oof_lgb_top):+.4f}')

10. LightGBM top-60, num_leaves=4                       AUC = 0.6594
10. LightGBM top-60, num_leaves=15                      AUC = 0.6597
10. LightGBM top-60, extra_trees                        AUC = 0.6548
10. LightGBM top-60, min_child_samples=300              AUC = 0.6615
10. LightGBM top-60, colsample=0.5                      AUC = 0.6611
10. LightGBM top-60, max_depth=3                        AUC = 0.6604


## 11. Tur × yo'nalish bo'yicha floor / ceiling featurelari (EDA 8-bo'lim)

In [13]:
tx = pd.read_parquet(f'{DATA}/train_transactions.parquet')
tx['grp'] = tx.tranzaksiya_turi + '_' + tx.kirim_chiqim
fl = tx.groupby('grp').miqdor_indeksi.min(); ce = tx.groupby('grp').miqdor_indeksi.max()
tx['fl'] = (tx.miqdor_indeksi < tx.grp.map(fl) + 0.02).astype(int); tx['ce'] = (tx.miqdor_indeksi > tx.grp.map(ce) - 0.02).astype(int)
E = pd.concat([tx.pivot_table(index='signal_id', columns='grp', values=c, aggfunc='sum').fillna(0).add_prefix(f'{c}_n_') for c in ['fl', 'ce']]
              + [tx.pivot_table(index='signal_id', columns='grp', values='miqdor_indeksi', aggfunc=a).add_prefix(f'g{a}_') for a in ['min', 'max']], axis=1)
E = E.reindex(tr.signal_id).astype('float32'); del tx
oof = cv_lgb(pd.concat([X_top, E], axis=1), early_stop=False)
record('11. Top-60 + floor/ceiling featurelari', roc_auc_score(y, oof), f'top-60 ga nisbatan {roc_auc_score(y, oof) - roc_auc_score(y, oof_lgb_top):+.4f}')

11. Top-60 + floor/ceiling featurelari                  AUC = 0.6627


## Natijalar jadvali

In [14]:
res = pd.read_csv(f'{OUT}/experiments.csv')
pd.set_option('display.max_colwidth', 80); display(res)

,key,Yondashuv,CV ROC-AUC,Xulosa
0,agg,1. Agregatlar (oynalar) + LightGBM,0.6271,bazaviy nuqta
1,agg_hist,2. + gistogramma/kvantil featurelari,0.6297,bazaviyga nisbatan +0.003
2,all853,3. + nisbiy tur/yo'nalish farqlari (853 feature),0.6520,oldingi qadamga nisbatan +0.022
3,lr853,4. Logistik regressiya (853 feature),0.6198,LightGBM (3-qadam) bilan farq -0.032
4,top30,5. Top-30 feature + LightGBM (1500 daraxt),0.6567,shovqinli featurelarni olib tashlash
5,top60,5. Top-60 feature + LightGBM (1500 daraxt),0.6618,shovqinli featurelarni olib tashlash
6,top120,5. Top-120 feature + LightGBM (1500 daraxt),0.6604,shovqinli featurelarni olib tashlash
7,lr60,6a. Logistik regressiya (top-60),0.6476,NaN
8,blend5,6b. Blend LightGBM 0.5 / LogReg 0.5,0.6588,NaN
9,blend7,6b. Blend LightGBM 0.7 / LogReg 0.3,0.6610,yakuniy retsept


## Xulosa (natijalardan avtomatik)

In [15]:
r = res.set_index('Yondashuv')['CV ROC-AUC']
g = lambda p: r[[k for k in r.index if k.startswith(p)]].iloc[0]
print(f"Featurelar:  bazaviy {g('1.'):.3f} → +gistogramma {g('2.'):.3f} → +nisbiy {g('3.'):.3f} → top-60 {g('5. Top-60'):.3f}")
print(f"Modellar (bir xil featurelar):  LightGBM {g('3.'):.3f} · LogReg {g('4.'):.3f} · CatBoost {g('7.'):.3f}")
print(f"Ketma-ketlik:  MIL {g('8.'):.3f} · GRU {g('9.'):.3f}")
print(f"Giperparametrlar oralig'i:  {r[[k for k in r.index if k.startswith('10.')]].min():.4f} … {r[[k for k in r.index if k.startswith('10.')]].max():.4f}")
print(f"Yakuniy retsept (blend 0.7/0.3):  {g('6b. Blend LightGBM 0.7'):.4f}")

Featurelar:  bazaviy 0.627 → +gistogramma 0.630 → +nisbiy 0.652 → top-60 0.662
Modellar (bir xil featurelar):  LightGBM 0.652 · LogReg 0.620 · CatBoost 0.647
Ketma-ketlik:  MIL 0.554 · GRU 0.626
Giperparametrlar oralig'i:  0.6548 … 0.6615
Yakuniy retsept (blend 0.7/0.3):  0.6610
